# Trump on social media, 2009 to 2026
**Knox data task, Atlantic Data Institute brief**
Author: Armaan Khan

This notebook takes the two raw files from Knox, cleans them, and builds one combined table that the rest of the analysis runs on.
The raw files are never changed. Everything cleaned is saved as a new file.

## 1. Loading the data

In [ ]:
# Connecting Google Drive so the notebook can read the files in My Drive/Knox
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/Knox'

In [2]:
# Libraries: pandas for tables, numpy for maths, html to fix text like &amp;, duckdb for SQL on the tables
import html
import numpy as np
import pandas as pd
import duckdb

In [3]:
# Loading both files. IDs are read as text so pandas doesn't round them
tweets_raw = pd.read_csv(f'{DATA_DIR}/trump_tweets.csv', dtype={'id': str})
truth_raw = pd.read_csv(f'{DATA_DIR}/trump_posts_full.csv', dtype={'id': str}, low_memory=False)

# Rows and columns in each file
print(tweets_raw.shape)
print(truth_raw.shape)

(56571, 9)
(32754, 40)


In [4]:
# First look at the Twitter file
tweets_raw.head()

,id,text,isRetweet,isDeleted,device,favorites,retweets,date,isFlagged
0,1698308935,Be sure to tune in and watch Donald Trump on L...,f,f,Twitter Web Client,939,519,5/4/2009 18:54,f
1,1701461182,Donald Trump will be appearing on The View tom...,f,f,Twitter Web Client,259,34,5/5/2009 1:00,f
2,1737479987,Donald Trump reads Top Ten Financial Tips on L...,f,f,Twitter Web Client,37,15,5/8/2009 13:38,f
3,1741160716,New Blog Post: Celebrity Apprentice Finale and...,f,f,Twitter Web Client,29,11,5/8/2009 20:40,f
4,1773561338,"""""""My persona will never be that of a wallflow...",f,f,Twitter Web Client,1877,1321,5/12/2009 14:07,f


In [5]:
# First look at the Truth Social file
truth_raw.head()

,id,created_at,text,url,retruth_count,like_count,reply_count,engagement_ratio,is_retruth,hour_of_day,...,countries_mentioned,companies_mentioned,mentions_person,toxicity_score,topic,rhetorical_device,target_entity,sentiment_toward_target,is_policy_related,has_text
0,107797156496908384,2022-02-14 15:54:32.528000+00:00,Get Ready! Your favorite President will see y...,https://truthsocial.com/@realDonaldTrump/10779...,49001,264075,33994,0.7609,False,15,...,[],[],[],0.0015,Personal,Boast,NaN,positive,False,True
1,108211822140637685,2022-04-28 21:29:28.207000+00:00,I’M BACK! #COVFEFE,https://truthsocial.com/@realDonaldTrump/10821...,123196,410837,60886,0.6906,False,21,...,[],"[""I\u2019M BACK""]",[],0.0145,Personal,Boast,NaN,positive,False,True
2,108217783188791701,2022-04-29 22:45:26.592000+00:00,Thank you to all of the GREAT and BEAUTIFUL Am...,https://truthsocial.com/@realDonaldTrump/10821...,47614,217254,19641,0.7636,False,22,...,"[""Truth""]","[""Haters"", ""Social Media"", ""BEAUTIFUL"", ""GREAT...",[],0.0009,Media,Praise,Truth Social,positive,False,True
3,108221053343991942,2022-04-30 12:37:05.206000+00:00,RT @catturd2Joe Biden is going to fly gas guzz...,https://truthsocial.com/@realDonaldTrump/10822...,0,0,0,0.0000,False,12,...,"[""Seattle""]","[""Air Force""]","[""Biden""]",0.8355,Other,Accusation,Joe Biden,negative,True,True
4,108221053884053060,2022-04-30 12:37:13.451000+00:00,@melmul,https://truthsocial.com/@realDonaldTrump/10822...,4302,21853,889,0.8081,False,12,...,[],"[""@melmul""]",[],0.0037,Other,Other,NaN,neutral,False,True


## 2. Cleaning the Twitter archive (May 2009 to January 2021)
Problems found and what I did about each:
1. **Broken IDs.** The file has been saved through Excel at some point, so many IDs were rounded into scientific notation (e.g. `1.34757E+18`). They can't be used to tell tweets apart, so I built my own ID.
2. **Dates** are in US month/day format and UTC. I read them with the exact format and converted them to US Eastern time, because "best time to post" only means something in the audience's time zone.
3. **Retweets** are other people's words and show 0 likes in this archive, so I flagged them and kept them out of performance scoring.
4. **Missing likes.** 761 of his own tweets show 0 likes, almost all from 2012 and 2013. That's a gap in how the data was collected, not real zeros, so I marked them as unknown.
5. **HTML codes** like `&amp;` were turned back into normal characters.
6. **One duplicated tweet** (same time and text) was dropped.

Also checked and fine: no missing text, and no missing or negative like counts. 105 tweets from 2011 to 2015 quote another account ("@user ..."), but they fall before the period I scored, so I left them as they were.

In [6]:
# Working on a copy so the raw table stays untouched
tweets = tweets_raw.copy()

# One tweet appears twice (same time and text), so the copy is dropped
print('Duplicate tweets:', tweets.duplicated(['date', 'text']).sum())
tweets = tweets.drop_duplicates(['date', 'text']).reset_index(drop=True)

# Proof the IDs are broken: how many are in scientific notation, and how many fake duplicates that creates
print('IDs in scientific notation:', tweets['id'].str.contains('E+', regex=False).sum())
print('Fake duplicate IDs:', tweets['id'].duplicated().sum())

# My own ID instead
tweets['post_id'] = 'tw_' + tweets.index.astype(str)

Duplicate tweets: 1
IDs in scientific notation: 56391
Fake duplicate IDs: 21117


In [7]:
# Dates: exact US format (month/day/year), stored in UTC, converted to US Eastern time
tweets['posted_at'] = (pd.to_datetime(tweets['date'], format='%m/%d/%Y %H:%M')
                         .dt.tz_localize('UTC').dt.tz_convert('America/New_York'))
print('Unreadable dates:', tweets['posted_at'].isna().sum())
print('First tweet:', tweets['posted_at'].min().date(), '| Last tweet:', tweets['posted_at'].max().date())

Unreadable dates: 0
First tweet: 2009-05-04 | Last tweet: 2021-01-08


In [8]:
# Fixing HTML codes in the text, e.g. &amp; becomes &
tweets['text'] = tweets['text'].fillna('').map(html.unescape)

# Retweets: flagged by the archive ('t') or starting with 'RT @'
tweets['is_repost'] = (tweets['isRetweet'] == 't') | tweets['text'].str.startswith('RT @')
print('Retweets:', tweets['is_repost'].sum(), 'of', len(tweets))
print('Median likes on retweets:', tweets.loc[tweets['is_repost'], 'favorites'].median())

Retweets: 10004 of 56570
Median likes on retweets: 0.0


In [9]:
# Missing likes: his own tweets showing 0 likes. Where do they sit?
zero_own = tweets[~tweets['is_repost'] & (tweets['favorites'] == 0)]
print(zero_own['posted_at'].dt.year.value_counts().sort_index())

# Marked as unknown (NaN) rather than 0, so they don't drag the averages down
tweets['likes'] = tweets['favorites'].astype(float)
tweets.loc[~tweets['is_repost'] & (tweets['favorites'] == 0), 'likes'] = np.nan
tweets['shares'] = tweets['retweets'].astype(float)

posted_at
2012     82
2013    638
2014      2
2016      2
2018      1
2019      6
2020     27
2021      3
Name: count, dtype: int64


In [10]:
# Deleted and flagged tweets are kept, with a True/False column for each
# (deleted tweets are left out of scoring later, because their likes stopped counting when they were deleted)
tweets['is_deleted'] = tweets['isDeleted'] == 't'
tweets['is_flagged'] = tweets['isFlagged'] == 't'
print('Deleted:', tweets['is_deleted'].sum(), '| Flagged by Twitter:', tweets['is_flagged'].sum())

Deleted: 1091 | Flagged by Twitter: 304


## 3. Cleaning the Truth Social archive (February 2022 to April 2026)
Problems found and what I did about each:
1. **The repost column is wrong.** `is_retruth` says False for every row, but 6,953 posts start with `RT @` followed by someone else's account and words, and three quarters of them show 0 likes, like retweets on Twitter. I rebuilt the flag myself.
2. **Posts with no words.** Some are only an image or video, some only a link. I kept them as their own post types, because how they perform is a finding, but they can't go into the word-based analysis.
3. **A useless column.** `emotion` is 'unknown' or blank on every row, so I dropped it.
4. **Times** are UTC, so I converted them to US Eastern like the tweets. A few timestamps have a different format, so I read them flexibly.
5. **Ready-made labels** (topic, rhetorical device, sentiment, toxicity) came with the file with no explanation of how they were made, so I checked a sample by hand before relying on them.

In [11]:
# Working on a copy
truth = truth_raw.copy()
truth['post_id'] = 'ts_' + truth['id']
truth['text'] = truth['text'].fillna('').map(html.unescape)

# Times: UTC to US Eastern (format='mixed' handles the few timestamps written differently)
truth['posted_at'] = pd.to_datetime(truth['created_at'], utc=True, format='mixed').dt.tz_convert('America/New_York')
print('Unreadable dates:', truth['posted_at'].isna().sum())
print('First post:', truth['posted_at'].min().date(), '| Last post:', truth['posted_at'].max().date())

Unreadable dates: 0
First post: 2022-02-14 | Last post: 2026-04-22


In [12]:
# The provided repost column says False for every row...
print(truth['is_retruth'].value_counts())

# ...but these posts start with 'RT @' and the rest of the text is someone else's
truth['is_repost'] = truth['text'].str.match(r'^RT[: ]')
print('Posts starting with RT @:', truth['is_repost'].sum())
print('Most reposted accounts:')
print(truth.loc[truth['is_repost'], 'text'].str.extract(r'^RT[: ]+@(\w+)')[0].value_counts().head(5))
print('Share with 0 likes: reposts', round((truth.loc[truth['is_repost'], 'like_count'] == 0).mean(), 2),
      '| his own posts', round((truth.loc[~truth['is_repost'], 'like_count'] == 0).mean(), 2))

is_retruth
False    32754
Name: count, dtype: int64
Posts starting with RT @: 6953
Most reposted accounts:
0
realDonaldTrump         376
TeamTrumpPRESIDENT      160
realDonaldTrumphttps    115
realamericasvoice       102
RSBN                     88
Name: count, dtype: int64
Share with 0 likes: reposts 0.76 | his own posts 0.0


In [13]:
# Post types: repost, image/video only (no text), link only, or his own words
link_only = truth['text'].str.fullmatch(r'\s*https?://\S+\s*')
truth['post_type'] = np.select(
    [truth['is_repost'], ~truth['has_text'], link_only],
    ['repost', 'media_only', 'link_only'],
    default='text')
print(truth['post_type'].value_counts())

post_type
text          14161
repost         6953
media_only     5935
link_only      5705
Name: count, dtype: int64


In [14]:
# 'emotion' is 'unknown' or blank on every row, so it tells us nothing
print(truth['emotion'].value_counts(dropna=False))
truth = truth.drop(columns=['emotion', 'is_retruth'])

# Likes, shares (ReTruths) and replies, with reposts' likes set to unknown like on Twitter
truth['likes'] = truth['like_count'].astype(float)
truth.loc[truth['is_repost'], 'likes'] = np.nan
truth['shares'] = truth['retruth_count'].astype(float)
truth['replies'] = truth['reply_count'].astype(float)

emotion
unknown    26819
NaN         5935
Name: count, dtype: int64


## 4. One combined table
Both platforms in the same shape, so every later step works on one table.
Twitter posts get the same post types as Truth Social (repost, link only, text) so they can be compared fairly.

In [15]:
# Post types for tweets, matching Truth Social
tw_link_only = tweets['text'].str.fullmatch(r'\s*https?://\S+\s*')
tweets['post_type'] = np.select([tweets['is_repost'], tw_link_only], ['repost', 'link_only'], default='text')
tweets['platform'] = 'Twitter'
tweets['replies'] = np.nan          # the Twitter archive has no reply counts
truth['platform'] = 'Truth Social'
truth['device'] = None
truth['is_deleted'] = False
truth['is_flagged'] = False

common = ['post_id', 'platform', 'posted_at', 'text', 'post_type', 'is_repost',
          'likes', 'shares', 'replies', 'device', 'is_deleted', 'is_flagged']
labels = ['topic', 'rhetorical_device', 'target_entity', 'sentiment_toward_target', 'sentiment_label',
          'sentiment_score', 'toxicity_score', 'cap_ratio', 'word_count', 'mentions_media', 'is_policy_related']

posts = pd.concat([tweets[common], truth[common + labels]], ignore_index=True)
print(posts.shape)
posts.groupby('platform').agg(posts=('post_id', 'size'), first=('posted_at', 'min'), last=('posted_at', 'max'))

(89324, 23)


,posts,first,last
platform,,,
Truth Social,32754,2022-02-14 10:54:32.528000-05:00,2026-04-22 23:02:45.507000-04:00
Twitter,56570,2009-05-04 14:54:00-04:00,2021-01-08 10:44:00-05:00


In [16]:
# Sanity check with SQL: post types per platform and how many have usable likes
duckdb.sql('''
    SELECT platform, post_type, COUNT(*) AS posts, COUNT(likes) AS posts_with_likes, MEDIAN(likes) AS median_likes
    FROM posts
    GROUP BY platform, post_type
    ORDER BY platform, posts DESC
''').df()

,platform,post_type,posts,posts_with_likes,median_likes
0,Truth Social,text,14161,14161,16548.0
1,Truth Social,repost,6953,0,NaN
2,Truth Social,media_only,5935,5935,11477.0
3,Truth Social,link_only,5705,5705,9548.0
4,Twitter,text,45325,44569,1201.0
5,Twitter,repost,10004,10004,0.0
6,Twitter,link_only,1241,1236,54129.0


In [17]:
# Saving the clean table as a new file. The raw files stay untouched
import os
os.makedirs(f'{DATA_DIR}/clean', exist_ok=True)
posts.to_pickle(f'{DATA_DIR}/clean/posts_clean.pkl')
print('Saved', len(posts), 'posts')

Saved 89324 posts


### Cleaning decisions in one place
| Issue | Rows | Decision |
|---|---|---|
| Twitter IDs rounded by Excel | ~21,000 fake duplicates | Built my own ID |
| Duplicated tweet | 1 | Dropped |
| Deleted tweets (likes stopped counting when deleted) | 1,091 | Kept and marked, left out of scoring |
| Retweets (other people's words, 0 likes) | 10,004 | Kept and counted, left out of scoring |
| Own tweets showing 0 likes (2012-13 gap) | 761 | Marked as unknown |
| Truth Social repost column wrong | 6,953 | Rebuilt from 'RT @' |
| Image/video-only and link-only posts | 5,935 / 5,705 | Kept as their own post types |
| 'emotion' column, 'unknown' or blank everywhere | 32,754 | Dropped |
| UTC times | all | Converted to US Eastern |

## 5. Scoring each post against his normal
Raw likes can't be compared across 17 years: his audience grew, the platforms changed size, and Truth Social is far smaller than Twitter.
So I compared every post with his own typical post in the same month on the same platform.

**Score = this post's likes ÷ his median likes that month.** A score of 1 means a normal post for him at the time, 2 means double and 0.5 means half.
I used the median rather than the average so a handful of viral posts couldn't distort what "typical" means.
I left reposts out, because they aren't his words, and deleted tweets, because their likes stopped counting when they were deleted.

In [18]:
# Only his own posts can be scored
own = posts[~posts['is_repost']].copy()

# His typical post each month on each platform
own['month'] = own['posted_at'].dt.strftime('%Y-%m')
own['typical_likes'] = own.groupby(['platform', 'month'])['likes'].transform('median')

# Score: 1.0 = a normal post for him that month, 2.0 = double
own['score'] = own['likes'] / own['typical_likes']

# Deleted tweets aren't scored: their like counts stopped when they were deleted, often within minutes
own.loc[own['is_deleted'], 'score'] = np.nan
own.groupby('platform')['score'].describe()

,count,mean,std,min,25%,50%,75%,max
platform,,,,,,,,
Truth Social,25801.0,1.176436,0.794089,0.000000,0.708163,1.000000,1.434336,16.676635
Twitter,45288.0,5.358475,84.993393,0.000049,0.644172,1.002637,1.684211,9784.555556


In [19]:
# Virality: shares per like, i.e. how often people pass a post on rather than just like it
own['share_rate'] = own['shares'] / own['likes']

# Conversation: replies per like (Truth Social only, the Twitter archive has no replies)
own['reply_rate'] = own['replies'] / own['likes']

# Useful date parts for later, all in US Eastern time
own['year'] = own['posted_at'].dt.year
own['hour'] = own['posted_at'].dt.hour
own['day'] = own['posted_at'].dt.strftime('%Y-%m-%d')

In [20]:
# Outliers: the same IQR check I used in my university coursework, run on the score
def find_outliers_IQR(data):
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    IQR = q3 - q1
    outliers = data[((data < (q1 - 1.5 * IQR)) | (data > (q3 + 1.5 * IQR)))]
    return outliers

outliers = find_outliers_IQR(own['score'].dropna())
print("number of outliers: " + str(len(outliers)))
print("share of scored posts:", round(len(outliers) / own['score'].notna().sum(), 3))

number of outliers: 7023
share of scored posts: 0.099


In [21]:
# The five biggest posts against his normal, to check they are real posts and not data errors
own.nlargest(5, 'score')[['platform', 'day', 'text', 'likes', 'typical_likes', 'score']]

,platform,day,text,likes,typical_likes,score
10660,Twitter,2013-08-31,"Be prepared, there is a small chance that our ...",176122.0,18.0,9784.555556
8011,Twitter,2013-05-08,"Sorry losers and haters, but my I.Q. is one of...",154973.0,22.0,7044.227273
14623,Twitter,2014-06-04,Are you allowed to impeach a president for gro...,231077.0,38.0,6080.973684
7515,Twitter,2013-04-21,NO MERCY TO TERRORISTS you dumb bastards!,90600.0,19.0,4768.421053
6262,Twitter,2013-03-06,It was just announced that @MacMiller’s song “...,68999.0,15.0,4599.933333


The check caught a problem. The biggest scores were all tweets from 2013 and 2014 with over 60,000 likes, in months where his typical tweet got about 20.
Those likes can't have come at the time. The archive must have counted likes years later, after he became famous, when people dug up his old tweets.
His typical likes only climbed into the hundreds after he launched his campaign on 16 June 2015:

In [22]:
# His typical likes each month in 2015
own[(own['platform'] == 'Twitter') & (own['year'] == 2015)].groupby('month')['likes'].median()

month
2015-01      31.0
2015-02      44.5
2015-03      41.0
2015-04      39.0
2015-05      42.0
2015-06     162.0
2015-07     659.0
2015-08    1610.0
2015-09    1899.5
2015-10    1494.0
2015-11    2009.5
2015-12    2984.0
Name: likes, dtype: float64

In [23]:
# So tweets from before July 2015 aren't scored
own.loc[(own['platform'] == 'Twitter') & (own['month'] < '2015-07'), 'score'] = np.nan

# Running the outlier check again
outliers = find_outliers_IQR(own['score'].dropna())
print("number of outliers: " + str(len(outliers)))
print("share of scored posts:", round(len(outliers) / own['score'].notna().sum(), 3))
own.nlargest(5, 'score')[['platform', 'day', 'text', 'likes', 'typical_likes', 'score']]

number of outliers: 2662
share of scored posts: 0.054


,platform,day,text,likes,typical_likes,score
25840,Twitter,2015-12-22,When I said that Hillary Clinton got schlonged...,73711.0,2984.0,24.702078
53505,Twitter,2020-10-02,"Tonight, @FLOTUS and I tested positive for COV...",1869706.0,81332.0,22.988565
28127,Twitter,2016-06-09,How long did it take your staff of 823 people ...,246342.0,12407.0,19.855082
23461,Twitter,2015-08-18,"When foreigners attend our great colleges &, w...",30261.0,1610.0,18.795652
81942,Truth Social,2025-03-04,"Please explain to Governor Trudeau, of Canada,...",92572.0,5551.0,16.676635


In [24]:
# Why the median: a few big posts pull the average above a typical post, and the spread is wide
duckdb.sql('''
    SELECT platform,
           COUNT(likes) AS posts,
           ROUND(AVG(likes)) AS mean_likes,
           MEDIAN(likes) AS median_likes,
           ROUND(STDDEV(likes)) AS std_likes,
           MAX(likes) - MIN(likes) AS range_likes,
           QUANTILE_CONT(likes, 0.75) - QUANTILE_CONT(likes, 0.25) AS iqr_likes
    FROM own
    WHERE score IS NOT NULL
    GROUP BY platform
''').df()

,platform,posts,mean_likes,median_likes,std_likes,range_likes,iqr_likes
0,Twitter,23580,67076.0,55571.5,73335.0,1869705.0,73918.25
1,Truth Social,25801,16138.0,13221.0,12018.0,410837.0,11307.00


After the fix, 5.4% of scored posts are outliers, all on the high side (a score can't go below 0, so nothing falls under the lower fence).
The biggest ones are real moments: his COVID diagnosis in October 2020, his statement after the Butler shooting in July 2024, and the "33,000 emails" reply to Hillary Clinton in 2016.
I kept them. They're genuine, and big moments are part of what the brief asks about. Every finding below uses medians, so a few huge posts can't pull the results around.

## 6. Tagging each post: style and topic
I tagged each post's style with simple rules that work the same on both platforms, so 2016 and 2026 could be compared fairly:
heavy capitals, exclamation marks, questions, links, nicknames for opponents, attacks on the media, and mentions of the other side.

In [25]:
# Word lists for the attack tags
nickname_words = (r"\b(?:crooked|sleepy|lyin'?|crazy|cryin'?|pocahontas|shifty|mini mike|low energy|little marco|sloppy|"
                  r"nervous nancy|tampon tim|comrade kamala|laffin'?|newscum|deranged|birdbrain|meatball ron|desanctimonious|rino)\b")
media_words = r"fake news|enemy of the people|lamestream|failing new york times|corrupt media|fake media"
other_side_words = r"\b(?:democrats?|radical left|the left|liberals?|leftists?|communists?|marxists?|socialists?|dems)\b"

In [26]:
# Style tags, the same rules for both platforms
text = own['text']
own['heavy_caps'] = text.str.count(r'\b[A-Z]{3,}\b') >= 3
own['exclamation'] = text.str.contains('!')
own['question'] = text.str.contains(r'\?')
own['has_link'] = text.str.contains('http')
own['nickname'] = text.str.contains(nickname_words, case=False)
own['media_attack'] = text.str.contains(media_words, case=False)
own['other_side'] = text.str.contains(other_side_words, case=False)

own.loc[own['post_type'] == 'text', ['heavy_caps', 'exclamation', 'question', 'has_link', 'nickname', 'media_attack', 'other_side']].mean().round(3)

heavy_caps      0.153
exclamation     0.585
question        0.079
has_link        0.247
nickname        0.050
media_attack    0.025
other_side      0.067
dtype: float64

**Topic tags.** The Truth Social file came with a topic label on every post, but the tweets have none.
To compare topics across both eras, I trained a simple text model on the labelled Truth Social posts and used it to label the tweets the same way.
I tested it on 20% of posts it never saw. I only used the topics it got right reliably, and left anything it wasn't sure about as "Other/unclear".

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Training data: Truth Social posts in his own words, with their first topic label. Rare topics go into Other
main_topics = ['Elections', 'Legal', 'Economy', 'Foreign Policy', 'Media', 'Personal', 'Immigration', 'Military', 'Religion']
labelled = own[(own['platform'] == 'Truth Social') & (own['post_type'] == 'text')].copy()
labelled['topic_main'] = labelled['topic'].fillna('Other').str.split('|').str[0]
labelled.loc[~labelled['topic_main'].isin(main_topics), 'topic_main'] = 'Other'
labelled['topic_main'].value_counts()

topic_main
Elections         3728
Legal             3055
Other             2056
Economy           1348
Foreign Policy    1138
Media              994
Personal           891
Immigration        596
Military           237
Religion           118
Name: count, dtype: int64

In [28]:
# Links are removed so the model learns from his words only
def words_only(s):
    return s.str.replace(r'http\S+', '', regex=True)

X_train, X_test, y_train, y_test = train_test_split(
    words_only(labelled['text']), labelled['topic_main'], test_size=0.2, random_state=42, stratify=labelled['topic_main'])

# TF-IDF turns each post into word weights, logistic regression learns which words go with which topic
vectoriser = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, stop_words='english')
model = LogisticRegression(max_iter=3000, C=4)
model.fit(vectoriser.fit_transform(X_train), y_train)

print(classification_report(y_test, model.predict(vectoriser.transform(X_test)), digits=2))

                precision    recall  f1-score   support

       Economy       0.76      0.69      0.72       270
     Elections       0.75      0.84      0.79       746
Foreign Policy       0.87      0.76      0.81       228
   Immigration       0.91      0.71      0.80       119
         Legal       0.87      0.87      0.87       611
         Media       0.73      0.70      0.71       199
      Military       0.62      0.17      0.27        47
         Other       0.48      0.61      0.54       411
      Personal       0.47      0.30      0.36       178
      Religion       0.67      0.17      0.27        24

      accuracy                           0.73      2833
     macro avg       0.71      0.58      0.61      2833
  weighted avg       0.73      0.73      0.72      2833



In [29]:
# Only topics the model gets right reliably (F1 of 0.70 or more on the test set) are used
reliable_topics = ['Elections', 'Legal', 'Economy', 'Foreign Policy', 'Immigration', 'Media']

# Labelling the tweets. If the model is less than 50% sure, the tweet stays 'Other/unclear'
is_tweet = (own['platform'] == 'Twitter') & (own['post_type'] == 'text')
probs = model.predict_proba(vectoriser.transform(words_only(own.loc[is_tweet, 'text'])))
guess = model.classes_[probs.argmax(axis=1)]
sure = probs.max(axis=1) >= 0.5
own.loc[is_tweet, 'topic_tag'] = np.where(sure & np.isin(guess, reliable_topics), guess, 'Other/unclear')

# Truth Social posts keep their own labels, grouped the same way
is_truth = (own['platform'] == 'Truth Social') & (own['post_type'] == 'text')
first_label = own.loc[is_truth, 'topic'].fillna('Other').str.split('|').str[0]
own.loc[is_truth, 'topic_tag'] = first_label.where(first_label.isin(reliable_topics), 'Other/unclear')

own.groupby('platform')['topic_tag'].value_counts(normalize=True).round(3)

platform      topic_tag     
Truth Social  Elections         0.263
              Other/unclear     0.233
              Legal             0.216
              Economy           0.095
              Foreign Policy    0.080
              Media             0.070
              Immigration       0.042
Twitter       Other/unclear     0.723
              Elections         0.110
              Legal             0.044
              Media             0.044
              Economy           0.037
              Foreign Policy    0.029
              Immigration       0.012
Name: proportion, dtype: float64

In [30]:
# Hand check: 50 Truth Social posts (are the labels that came with the file right?) and 30 tweets (is the model right?)
truth_sample = own[is_truth & own['text'].str.len().between(60, 400)].sample(50, random_state=7)
tweet_sample = own[is_tweet & (own['year'] >= 2015) & (own['topic_tag'] != 'Other/unclear')].sample(30, random_state=7)

hand_check = pd.concat([
    truth_sample.assign(label_to_check='Topic: ' + truth_sample['topic'].fillna('') + ' | Device: ' + truth_sample['rhetorical_device'].fillna('')),
    tweet_sample.assign(label_to_check='Topic (model): ' + tweet_sample['topic_tag'])])
hand_check = hand_check[['post_id', 'platform', 'day', 'text', 'label_to_check']].assign(topic_correct='', tone_correct='', notes='')
hand_check.to_csv(f'{DATA_DIR}/hand_check.csv', index=False)
print(len(hand_check), 'posts saved for checking by hand')

80 posts saved for checking by hand


**Hand check results.** I checked the 80 posts in Excel: Y if the label was right, N if it was wrong, and ? if I wasn't sure. The ? answers are left out of the counts.

In [31]:
# The hand check, filled in by hand in Excel (the first row of the sheet is instructions)
checked = pd.read_excel(f'{DATA_DIR}/hand_check_results.xlsx', header=1)
checked = checked.rename(columns={'topic_correct (Y/N)': 'topic_correct', 'tone_correct (Y/N, Truth Social only)': 'tone_correct'})

# Answers were typed by hand, so tidy them first
for col in ['topic_correct', 'tone_correct']:
    checked[col] = checked[col].astype(str).str.strip().str.upper()

def share_right(answers):
    answers = answers[answers.isin(['Y', 'N'])]
    return f"{(answers == 'Y').sum()} of {len(answers)} right ({(answers == 'Y').mean():.0%})"

truth_checked = checked[checked['platform'] == 'Truth Social']
tweet_checked = checked[checked['platform'] == 'Twitter']
first_topic = truth_checked['label_to_check'].str.extract(r'Topic: ([^|]*)')[0].str.strip()

print('Truth Social topic labels:          ', share_right(truth_checked['topic_correct']))
print('...on the six topics the report uses:', share_right(truth_checked.loc[first_topic.isin(reliable_topics), 'topic_correct']))
print('Truth Social tone labels:           ', share_right(truth_checked['tone_correct']))
print('Tweet topics from my model:         ', share_right(tweet_checked['topic_correct']))

Truth Social topic labels:           43 of 47 right (91%)
...on the six topics the report uses: 37 of 37 right (100%)
Truth Social tone labels:            41 of 48 right (85%)
Tweet topics from my model:          25 of 30 right (83%)


In [32]:
# Which tone labels were wrong most often
truth_checked = truth_checked.assign(tone_label=truth_checked['label_to_check'].str.extract(r'Device: (.*)$')[0].str.strip())
truth_checked.groupby('tone_label')['tone_correct'].value_counts().unstack(fill_value=0)

tone_correct,?,N,Y
tone_label,,,
Accusation,1,0,15
Boast,0,1,9
Call to Action,0,0,2
Denial,0,0,2
Insult,1,0,3
Other,0,1,0
Policy Statement,0,0,2
Praise,0,5,8


The topic labels held up: every checked post on the six topics I used was labelled correctly, and the four mistakes were all in "Other" or "Personal", which the report doesn't use.
The tone labels were mostly right, but Praise was the weakest. 5 of the 13 posts labelled Praise were really something else, mostly him praising himself (a boast). So I kept the tone chart but flagged the praise result in the report.
My model's topics for tweets were right 5 times in 6, with Media the weakest: some tweets about elections were labelled Media.
This was a quick check of 80 posts by one person, so the percentages are rough. It's enough to show whether the labels can be trusted, which was the point.

## 7. The analysis
I ran SQL (DuckDB) directly on the `own` table. Each query answers one part of the brief.
Twitter scores start in July 2015, when his campaign began (see section 5). Where a query says 2017 onwards, it compares his presidency with Truth Social.

In [33]:
# 7.1 Seventeen years in one view: posts per month, his own words and reposts
timeline = duckdb.sql('''
    SELECT platform, strftime(posted_at, '%Y-%m') AS month,
           SUM(CASE WHEN is_repost THEN 0 ELSE 1 END) AS own_posts,
           SUM(CASE WHEN is_repost THEN 1 ELSE 0 END) AS reposts
    FROM posts
    GROUP BY platform, month
    ORDER BY month
''').df()
timeline.tail()

,platform,month,own_posts,reposts
186,Truth Social,2025-12,365.0,111.0
187,Truth Social,2026-01,483.0,123.0
188,Truth Social,2026-02,452.0,103.0
189,Truth Social,2026-03,501.0,37.0
190,Truth Social,2026-04,390.0,55.0


In [34]:
# 7.2 What kind of post works: his own words vs link only vs image/video only
post_types = duckdb.sql('''
    SELECT platform, post_type, COUNT(*) AS posts, ROUND(MEDIAN(score), 2) AS median_score
    FROM own
    WHERE score IS NOT NULL
    GROUP BY platform, post_type
    ORDER BY platform, median_score DESC
''').df()
post_types

,platform,post_type,posts,median_score
0,Truth Social,text,14161,1.23
1,Truth Social,media_only,5935,0.89
2,Truth Social,link_only,5705,0.74
3,Twitter,text,22353,1.03
4,Twitter,link_only,1227,0.68


In [35]:
# 7.3 Style: median score with and without each tag, on both platforms (posts in his own words)
style_tags = ['heavy_caps', 'exclamation', 'question', 'has_link', 'nickname', 'media_attack', 'other_side']
tag_results = []
for tag in style_tags:
    tag_results.append(duckdb.sql(f'''
        SELECT platform, '{tag}' AS tag, {tag} AS tagged, COUNT(*) AS posts, ROUND(MEDIAN(score), 2) AS median_score
        FROM own
        WHERE post_type = 'text' AND score IS NOT NULL
        GROUP BY platform, {tag}
        ORDER BY platform, {tag}
    ''').df())

styles = pd.concat(tag_results, ignore_index=True)
styles.pivot_table(index='tag', columns=['platform', 'tagged'], values='median_score')

platform     Truth Social       Twitter      
tagged              False True    False True 
tag                                          
exclamation          1.03  1.35    0.96  1.06
has_link             1.33  0.95    1.13  0.83
heavy_caps           1.15  1.38    1.01  1.14
media_attack         1.22  1.53    1.02  1.18
nickname             1.20  1.47    1.01  1.22
other_side           1.21  1.40    1.01  1.12
question             1.22  1.39    1.01  1.20

In [36]:
# 7.4 Topics: how often he posts about each one vs how well it does
topics = duckdb.sql('''
    SELECT platform, topic_tag,
           COUNT(*) AS posts,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY platform), 1) AS pct_of_posts,
           ROUND(MEDIAN(score), 2) AS median_score,
           ROUND(MEDIAN(share_rate), 3) AS median_share_rate
    FROM own
    WHERE post_type = 'text' AND score IS NOT NULL AND (platform = 'Truth Social' OR year >= 2017)
    GROUP BY platform, topic_tag
    ORDER BY platform, median_score DESC
''').df()
topics

,platform,topic_tag,posts,pct_of_posts,median_score,median_share_rate
0,Truth Social,Foreign Policy,1138,8.0,1.42,0.237
1,Truth Social,Immigration,596,4.2,1.35,0.272
2,Truth Social,Legal,3055,21.6,1.33,0.279
3,Truth Social,Other/unclear,3302,23.3,1.27,0.243
4,Truth Social,Media,994,7.0,1.20,0.248
5,Truth Social,Economy,1348,9.5,1.20,0.254
6,Truth Social,Elections,3728,26.3,1.09,0.249
7,Twitter,Immigration,385,2.6,1.13,0.226
8,Twitter,Legal,1398,9.4,1.11,0.240
9,Twitter,Foreign Policy,719,4.9,1.09,0.210


In [37]:
# 7.5 Truth Social only: the tone labels that came with the file (attack, praise, boast...)
devices = duckdb.sql('''
    SELECT rhetorical_device, COUNT(*) AS posts, ROUND(MEDIAN(score), 2) AS median_score,
           ROUND(MEDIAN(reply_rate), 3) AS median_reply_rate
    FROM own
    WHERE platform = 'Truth Social' AND post_type = 'text' AND score IS NOT NULL
    GROUP BY rhetorical_device
    HAVING COUNT(*) >= 100
    ORDER BY median_score DESC
''').df()
devices

,rhetorical_device,posts,median_score,median_reply_rate
0,Threat,171,1.68,0.087
1,Policy Statement,1178,1.41,0.061
2,Denial,460,1.39,0.054
3,Boast,1619,1.35,0.055
4,Accusation,3583,1.35,0.059
5,Insult,1067,1.31,0.063
6,Call to Action,1321,1.21,0.053
7,Other,1159,1.11,0.054
8,Praise,3469,1.02,0.033


In [38]:
# 7.6 Posting volume: does each post do worse on days he posts a lot?
volume = duckdb.sql('''
    WITH daily AS (
        SELECT platform, day, score, COUNT(*) OVER (PARTITION BY platform, day) AS posts_that_day
        FROM own
        WHERE score IS NOT NULL AND year >= 2017
    )
    SELECT platform,
           CASE WHEN posts_that_day <= 3 THEN '1-3'
                WHEN posts_that_day <= 7 THEN '4-7'
                WHEN posts_that_day <= 15 THEN '8-15'
                WHEN posts_that_day <= 30 THEN '16-30'
                ELSE '31+' END AS posts_per_day,
           COUNT(DISTINCT day) AS days, ROUND(MEDIAN(score), 2) AS median_score
    FROM daily
    GROUP BY platform, posts_per_day
    ORDER BY platform, MIN(posts_that_day)
''').df()
volume

,platform,posts_per_day,days,median_score
0,Truth Social,1-3,75,1.64
1,Truth Social,4-7,217,1.41
2,Truth Social,8-15,489,1.24
3,Truth Social,16-30,469,1.01
4,Truth Social,31+,199,0.80
5,Twitter,1-3,179,1.28
6,Twitter,4-7,381,1.12
7,Twitter,8-15,556,1.03
8,Twitter,16-30,302,0.97
9,Twitter,31+,37,0.78


In [39]:
# 7.7 Timing: median score by hour of day, US Eastern time (hours with under 100 posts left out, too few to trust)
timing = duckdb.sql('''
    SELECT platform, hour, COUNT(*) AS posts, ROUND(MEDIAN(score), 2) AS median_score
    FROM own
    WHERE score IS NOT NULL AND (platform = 'Truth Social' OR year >= 2017)
    GROUP BY platform, hour
    HAVING COUNT(*) >= 100
    ORDER BY platform, hour
''').df()
timing.pivot(index='hour', columns='platform', values='median_score').T

hour,0,1,2,6,7,8,9,10,11,12,...,14,15,16,17,18,19,20,21,22,23
platform,,,,,,,,,,,,,,,,,,,,,
Truth Social,1.17,1.14,1.2,1.19,1.32,1.23,1.13,0.86,0.80,0.82,...,0.91,0.92,0.93,1.04,1.1,1.05,1.02,1.01,1.15,1.13
Twitter,0.94,NaN,NaN,1.04,1.05,1.08,1.10,1.11,1.02,0.95,...,0.99,0.94,0.95,0.97,1.0,0.99,1.00,0.99,0.97,0.97


In [40]:
# 7.8 Repetition: how election posts have performed year by year
elections = duckdb.sql('''
    SELECT platform, year,
           ROUND(AVG(CASE WHEN topic_tag = 'Elections' THEN 1 ELSE 0 END) * 100, 1) AS pct_election_posts,
           ROUND(MEDIAN(CASE WHEN topic_tag = 'Elections' THEN score END), 2) AS election_score,
           ROUND(MEDIAN(CASE WHEN topic_tag <> 'Elections' THEN score END), 2) AS other_score
    FROM own
    WHERE post_type = 'text' AND score IS NOT NULL
    GROUP BY platform, year
    ORDER BY platform, year
''').df()
elections

,platform,year,pct_election_posts,election_score,other_score
0,Truth Social,2022,35.8,1.20,1.27
1,Truth Social,2023,25.1,1.24,1.40
2,Truth Social,2024,34.2,1.05,1.22
3,Truth Social,2025,12.3,1.00,1.29
4,Truth Social,2026,21.8,0.74,1.34
5,Twitter,2015,17.8,1.10,0.98
6,Twitter,2016,22.9,0.99,1.00
7,Twitter,2017,6.2,0.96,1.01
8,Twitter,2018,11.5,0.86,1.04
9,Twitter,2019,7.8,0.98,1.03


In [41]:
# 7.9 A scorecard modelled on the Knox Index: his first term on Twitter vs his second term on Truth Social
scorecard = duckdb.sql('''
    WITH periods AS (
        SELECT *,
               CASE WHEN platform = 'Twitter' AND year BETWEEN 2017 AND 2020 THEN 'Twitter 2017-20'
                    WHEN platform = 'Truth Social' AND year >= 2025 THEN 'Truth Social 2025-26'
               END AS period
        FROM own
    )
    SELECT period,
           ROUND(COUNT(*) / COUNT(DISTINCT day), 1) AS posts_per_day,
           MEDIAN(likes) AS median_likes,
           ROUND(MEDIAN(share_rate), 3) AS shares_per_like
    FROM periods
    WHERE period IS NOT NULL
    GROUP BY period
    ORDER BY period DESC
''').df()

# Followers aren't in the data, so these come from news reports (sources in the web report)
# Twitter: 88.7 million when he was banned in January 2021. Truth Social: "more than 10 million" in mid-2025
scorecard['followers'] = scorecard['period'].map({'Twitter 2017-20': 88.7e6, 'Truth Social 2025-26': 10e6})
scorecard['engagement_pct'] = (scorecard['median_likes'] / scorecard['followers'] * 100).round(3)
scorecard

,period,posts_per_day,median_likes,shares_per_like,followers,engagement_pct
0,Twitter 2017-20,11.4,72753.5,0.224,88700000.0,0.082
1,Truth Social 2025-26,15.9,13545.0,0.240,10000000.0,0.135


### Do the findings hold up when looked at together?
Some tags overlap (an attack is often also in capitals), so I fitted one model that looks at them all at once, plus topic, posting volume and hour.
Each number is how much that feature changes a post's score with everything else held equal.

In [42]:
import statsmodels.formula.api as smf

# Truth Social posts in his own words, with how many posts he made that day
check = own[(own['platform'] == 'Truth Social') & (own['post_type'] == 'text') & (own['score'] > 0)].copy()
check['posts_that_day'] = check.groupby('day')['post_id'].transform('count')
check['log_score'] = np.log(check['score'])

result = smf.ols('log_score ~ heavy_caps + exclamation + question + has_link + nickname + media_attack + other_side'
                 ' + C(topic_tag) + np.log(posts_that_day) + C(hour)', data=check).fit()

# Turning the model's numbers into "% change in score", keeping only the style tags and volume
effects = (np.exp(result.params) - 1).mul(100).round(1)
effects[[i for i in effects.index if 'True' in i or 'posts_that_day' in i]]

heavy_caps[T.True]         7.1
exclamation[T.True]       12.6
question[T.True]           4.2
has_link[T.True]         -20.6
nickname[T.True]          13.3
media_attack[T.True]       9.0
other_side[T.True]         5.2
np.log(posts_that_day)   -22.9
dtype: float64

## 8. Exporting the numbers for the web report
I exported everything the charts need into one small JSON file, so the page itself never touches the raw data.

In [43]:
import json

def records(df):
    return json.loads(df.to_json(orient='records'))

report_data = {
    'timeline': records(timeline),
    'post_types': records(post_types),
    'styles': records(styles),
    'topics': records(topics),
    'devices': records(devices),
    'volume': records(volume),
    'timing': records(timing),
    'elections': records(elections),
    'scorecard': records(scorecard),
    'effects': effects.to_dict(),
    'counts': {'tweets': int((posts['platform'] == 'Twitter').sum()), 'truths': int((posts['platform'] == 'Truth Social').sum()),
               'own_scored': int(own['score'].notna().sum())},
}
with open(f'{DATA_DIR}/report_data.json', 'w') as f:
    json.dump(report_data, f, indent=1)
print('Saved report_data.json')

Saved report_data.json
